# k3 — Mã hoá BGE-M3 đoạn cha + đoạn con (GPU T4 ×2, chia cho nhiều người)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`
(+ output phiên trước của CHÍNH người này nếu bị ngắt).

**Chia việc:** `MEMBERS` người × 2 GPU. Người `m` đặt `MEMBER = m`; GPU g chạy phần `2m+g / 2·MEMBERS`. Mọi người phải
dùng **cùng bản `mir2-chunks`**: dòng "dấu vân tay" in ra phải giống nhau.
**Mã hoá bổ sung** (kho thêm bài, vd longchau): Add Input các dataset emb CŨ ĐẦY ĐỦ (vd `mir2-emb-m0..m2` bản trước) và
ghi tên chúng vào `SKIP_FROM` → chỉ mã hoá đoạn mới / đổi nội dung (theo cặp unit_id + hash). Mọi người phải dùng CÙNG
`SKIP_FROM`. Output dở dang của chính mình (để chạy tiếp) KHÔNG ghi vào `SKIP_FROM` — nó tự được dùng để chép khối đã xong.
Ước tính (chưa đo trên kho thật): đoạn cha ~2,6 triệu, đoạn con ~6,5 triệu; tổng ~8–10 giờ GPU ×2 → ~3 giờ/người khi 3 người.
**Đầu ra:** `emb/{parents,children}/<khối>/` → dataset **`mir2-emb-m{MEMBER}`**.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k3-v1 (15/10/2026)'
MEMBER = None        # ← 0, 1, 2 …
MEMBERS = 3
LEVELS = ['parents', 'children']
SKIP_FROM = []       # tên dataset emb cũ đầy đủ (vd ['mir2-emb-m0-v1']) — giống nhau ở mọi người
RUN_BENCH = False
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) Thư mục emb của phiên trước / bản cũ trong /kaggle/input
assert MEMBER is not None and 0 <= MEMBER < MEMBERS, 'Đặt MEMBER ở ô (1)'
prev = [str(d / 'emb') for d in locate_all('emb')]
print('emb trong input:', prev)
old = [p for p in prev if any(f'/{x}/' in p + '/' for x in SKIP_FROM)]
mine_prev = [p for p in prev if p not in old]
skip = f' --skip {",".join(old)}' if old else ''
resume = f' --resume {",".join(mine_prev)}' if mine_prev else ''
print('bỏ qua (bản cũ):', old, '| chạy tiếp từ:', mine_prev)
if RUN_BENCH:
    sh(f'python -m mirret.encode bench --config {K.cfg} --level parents --n 2000 --device cuda:0')
for lv in LEVELS:
    sh(f'python -m mirret.encode plan --config {K.cfg} --level {lv}{skip}')

In [ ]:
# (3) Mã hoá phần của MEMBER trên 2 GPU, lần lượt từng mức
n = 2 * MEMBERS
for lv in LEVELS:
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.encode run --config {K.cfg} --level {lv} '
         f'--shard {2 * MEMBER + g}/{n} --device cuda:0{skip}{resume}' for g in range(2)], f'encode_{lv}')
report(f'k3_m{MEMBER}')